# ⚡ Tactical & Timing Models
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Realosunboy6/free-portfolio-visualizer/blob/main/notebooks/06_tactical_models.ipynb)

Moving-average timing (including crossover and multi-period weighted signals), relative strength, dual momentum, target volatility, seasonal, and adaptive allocation. Every signal is evaluated **with a one-period execution lag** (no look-ahead) and transaction costs — treat these as experiments, not oracles.

In [ ]:
#@title Setup — run this first {display-mode: "form"}
try:
    import portlab
except ImportError:
    %pip install -q "portlab @ git+https://github.com/Realosunboy6/free-portfolio-visualizer.git"
    import portlab
print("portlab", portlab.__version__, "ready")

In [ ]:
#@title Settings {display-mode: "form"}
risk_assets = "SPY, EFA, EEM, TLT, GLD"  #@param {type:"string"}
cash_asset = "BIL"          #@param {type:"string"}
start_date = "2008-01-01"   #@param {type:"date"}
ma_window = 200             #@param {type:"number"}
momentum_lookback = 126     #@param {type:"number"}
assets_to_hold = 2          #@param {type:"number"}
target_vol = 0.10           #@param {type:"number"}
transaction_cost_bps = 5    #@param {type:"number"}
SMOKE = False

In [ ]:
import pandas as pd
from portlab import metrics, plots, tactical
from portlab.data import get_prices
from portlab.returns import simple_returns

assets = [t.strip().upper() for t in risk_assets.split(",") if t.strip()]
cash = cash_asset.strip().upper()
prices = get_prices(assets + [cash], start_date)
rets = simple_returns(prices).fillna(0.0)
risk_prices = prices[assets]

strategies = {
  "Buy & Hold EW": pd.DataFrame(1/len(assets), index=prices.index, columns=assets),
  f"MA{ma_window} Timing": tactical.ma_timing(risk_prices, windows=ma_window, out_asset=cash),
  "MA Multi-Period": tactical.ma_timing(risk_prices,
        windows={50: 0.25, 100: 0.25, 150: 0.25, 200: 0.25}, out_asset=cash),
  f"Rel Strength Top{assets_to_hold}": tactical.relative_strength(
        risk_prices, lookbacks=momentum_lookback, top_n=assets_to_hold,
        out_asset=cash, ma_risk_control=ma_window),
  "Dual Momentum": tactical.dual_momentum(risk_prices, prices[cash],
        lookback=momentum_lookback, top_n=assets_to_hold, out_asset=cash),
  "Target Vol": tactical.target_volatility(rets,
        {a: 1/len(assets) for a in assets}, target_annual=target_vol, out_asset=cash),
  "Seasonal (Nov-Apr)": tactical.seasonal(prices.index, assets, out_asset=cash),
  "Adaptive Allocation": tactical.adaptive_allocation(
        risk_prices, rets[assets], lookback=momentum_lookback,
        top_n=assets_to_hold, out_asset=cash),
}
strat_rets = pd.DataFrame({n: tactical.evaluate(w, rets, tc_bps=transaction_cost_bps)
                           for n, w in strategies.items()}).dropna()
summary = pd.concat([metrics.summary(strat_rets[c], name=c) for c in strat_rets], axis=1)
summary.style.format("{:.3f}")

In [ ]:
plots.growth_chart(strat_rets, initial=10000, log_scale=True).show()
plots.drawdown_chart(strat_rets[["Buy & Hold EW", "Dual Momentum", f"MA{ma_window} Timing"]]).show()

In [ ]:
# Current signals — what would each model hold RIGHT NOW?
latest = pd.DataFrame({n: w.iloc[-1] for n, w in strategies.items()}).fillna(0.0)
latest.style.format("{:.1%}")